In [0]:
%sql

DECLARE OR REPLACE VARIABLE question STRING
DEFAULT 'What security features does RP2350 provide?';

WITH retrieved AS (
    SELECT
        file_name,
        page_number,
        product,
        document_type,
        chunk_id,
        text,
        search_score
    FROM VECTOR_SEARCH(
        index => 'microcontrollers.rag_project.document_search',
        query_text => question,
        num_results => 5
    )
),

context AS (
    SELECT
        concat_ws(
            '\n\n',
            collect_list(
                concat(
                    'Source: ', file_name,
                    '\nPage: ', CAST(page_number AS STRING),
                    '\nProduct: ', product,
                    '\nDocument type: ', document_type,
                    '\nContent: ', text
                )
            )
        ) AS retrieved_context
    FROM retrieved
)

SELECT ai_query(
    'databricks-gpt-oss-20b',
    concat(
        'You are a documentation question-answering assistant.

Answer the user question using ONLY the retrieved documentation.

Rules:
1. Use only information explicitly supported by the retrieved documentation.
2. Do not use your general knowledge.
3. Do not make assumptions or infer missing specifications.
4. If the retrieved documentation does not contain enough information to answer the question, say:
"The retrieved documentation does not contain enough information to answer this question."
5. Every factual claim must be supported by the retrieved documentation.
6. Do not invent citations, document names, page numbers, specifications, or features.
7. At the end of the answer, provide a "Sources:" section.
8. In the Sources section, list the document filename and page number for the documents that support the answer.
9. Do not put citation markers such as 【】 in the answer.
10. Keep the answer concise and directly answer the question.


User question:
',
        question,
        '

Retrieved documentation:
',
        retrieved_context
    )
) AS response
FROM context;


In [0]:
%sql
-- ============================================================
-- RAG QUESTION
-- Change ONLY this value for each new question
-- ============================================================

WITH params AS (
  SELECT 'How much SRAM does RP2040 have?' AS question
),

-- ============================================================
-- 1. Retrieve the most relevant document chunks
-- ============================================================

retrieved_chunks AS (
  SELECT
    file_name,
    page_number,
    product,
    document_type,
    chunk_id,
    text,
    search_score
  FROM VECTOR_SEARCH(
    index => 'microcontrollers.rag_project.document_search',
    query_text => (SELECT question FROM params),
    num_results => 5
  )
),

-- ============================================================
-- 2. Combine the retrieved chunks into LLM context
-- ============================================================

context AS (
  SELECT
    CONCAT_WS(
      '\n\n--- SOURCE ---\n',
      COLLECT_LIST(
        CONCAT(
          'Document: ', file_name,
          '\nPage: ', CAST(page_number AS STRING),
          '\nProduct: ', product,
          '\nDocument type: ', document_type,
          '\nContent:\n', text
        )
      )
    ) AS retrieved_context
  FROM retrieved_chunks
)

-- ============================================================
-- 3. Ask the LLM to answer using ONLY retrieved context
-- ============================================================

SELECT
  ai_query(
    'databricks-gpt-oss-20b',
    CONCAT(
      'You are a technical documentation question-answering assistant.

',
      'Your task is to answer the USER QUESTION using ONLY the RETRIEVED DOCUMENTATION.

',
      'STRICT GROUNDING RULES:

',
      '1. A retrieved chunk is NOT automatically evidence for the answer.
',
      '2. First determine whether the retrieved documentation actually contains ',
      'information that directly answers the user question.
',
      '3. Do NOT infer the meaning of numbers, names, identifiers, page numbers, ',
      'table-of-contents entries, or other text unless the documentation explicitly ',
      'establishes their meaning.
',
      '4. Do NOT treat a number as a price, date, specification, quantity, or other ',
      'measurement unless the retrieved documentation explicitly identifies it as such.
',
      '5. Do NOT use outside knowledge.
',
      '6. Do NOT fill gaps using assumptions or general knowledge.
',
      '7. Every factual claim in the answer must be directly supported by the retrieved ',
      'documentation.
',
      '8. If the retrieved documentation does not directly contain enough information ',
      'to answer the question, respond exactly:
',
      'The retrieved documentation does not contain enough information to answer this question.
',
      '9. Cite the source document filename and page number for factual claims.
',
      '10. Keep the answer concise.
',
    '11. Use the minimum number of source citations necessary to support the answer.
',
    '12. Prefer the most directly relevant page or pages over redundant citations.
',
      'USER QUESTION:
',
      (SELECT question FROM params),
      '

',
      'RETRIEVED DOCUMENTATION:
',
      retrieved_context
    )
  ) AS response
FROM context;




In [0]:
%sql
WITH params AS (
  SELECT :question AS question
),

retrieved_chunks AS (
  SELECT
    file_name,
    page_number,
    product,
    document_type,
    chunk_id,
    text,
    search_score
  FROM VECTOR_SEARCH(
    index => 'microcontrollers.rag_project.document_search',
    query_text => (SELECT question FROM params),
    num_results => 5
  )
),

context AS (
  SELECT
    CONCAT_WS(
      '\n\n--- SOURCE ---\n',
      COLLECT_LIST(
        CONCAT(
          'Document: ', file_name,
          '\nPage: ', CAST(page_number AS STRING),
          '\nProduct: ', product,
          '\nDocument type: ', document_type,
          '\nContent:\n', text
        )
      )
    ) AS retrieved_context
  FROM retrieved_chunks
)

SELECT
  ai_query(
    'databricks-gpt-oss-20b',
    CONCAT(
      'You are a technical documentation question-answering assistant.\n\n',

      'Answer the user question using ONLY the retrieved documentation.\n\n',

      'STRICT GROUNDING RULES:\n',
      '1. A retrieved chunk is not automatically evidence for the answer.\n',
      '2. Only use information that directly answers the user question.\n',
      '3. Do not use outside knowledge.\n',
      '4. Do not infer or invent technical details.\n',
      '5. Do not interpret unexplained numbers or identifiers.\n',
      '6. Every factual claim must be supported by the retrieved documentation.\n',
      
      '7. Use the minimum number of source citations necessary to support the answer.\n',
      '8. Prefer the most directly relevant page or pages over redundant citations.\n',
      '9. Cite the document filename and page number for factual claims.\n',
      
      '10. If the retrieved documentation does not contain enough information, ',
      'respond exactly: The retrieved documentation does not contain enough ',
      'information to answer this question.\n',
      
      '11. Keep the answer concise and technically accurate.\n\n',

      'USER QUESTION:\n',
      (SELECT question FROM params),
      '\n\n',

      'RETRIEVED DOCUMENTATION:\n',
      retrieved_context
    )
  ) AS response
FROM context;
